# Circuit Post-Hoc Analysis

Loads the discovered circuit (binary masks, no training) and runs the test set for detailed analysis.

In [ ]:
import os
import pickle
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, LlamaForCausalLM

# Change to the project root if running from elsewhere
os.chdir('/home/exouser/circuit_pruning')

from models.llama_circuit import PrunableLlamaForCausalLM, PruningConfig
from models.l0 import HardConcreteGate
from dataset.ioi_llama import IOIDatasetLlama
from utils import analyze_and_finalize_circuit, disable_dropout

## Configuration — set SAVE_DIR to your run's checkpoint folder

In [ ]:
SAVE_DIR = 'checkpoints_llama_kl_budget'  # <-- change if you used --save-dir
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load run config saved by the training script
with open(os.path.join(SAVE_DIR, 'run_config.pkl'), 'rb') as f:
    run_config = pickle.load(f)

MODEL_NAME = run_config['model']
print(f'Model : {MODEL_NAME}')
print(f'Device: {DEVICE}')
print(f'KL budget used: {run_config["kl_budget"]}')

## Load tokenizer & models

In [ ]:
HF_TOKEN = None  # set if needed, or put token string here
token_file = 'hf_tokken.txt'
if HF_TOKEN is None and os.path.exists(token_file):
    with open(token_file) as f:
        HF_TOKEN = f.read().strip()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {'token': HF_TOKEN, 'torch_dtype': torch.bfloat16}

# Full (reference) model — frozen
full_model = LlamaForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs).to(DEVICE).eval()
for p in full_model.parameters():
    p.requires_grad = False
print('Full model loaded.')

## Load circuit model and apply saved binary masks

In [ ]:
# We reuse the same PruningConfig used during training
from ioi_llama_kl_budget import KLBudgetLlamaPruningConfig

pruning_config = KLBudgetLlamaPruningConfig()
circuit_model = PrunableLlamaForCausalLM.from_pretrained_with_pruning(
    MODEL_NAME, pruning_config, **model_kwargs
).to(DEVICE).eval()
disable_dropout(circuit_model)

# Load the saved gate log_alphas and inject them
gate_log_alphas = torch.load(os.path.join(SAVE_DIR, 'gate_log_alphas.pt'), map_location='cpu')

loaded = 0
for name, module in circuit_model.named_modules():
    if isinstance(module, HardConcreteGate) and name in gate_log_alphas:
        module.log_alpha.data.copy_(gate_log_alphas[name].float())
        loaded += 1

print(f'Loaded {loaded} / {len(gate_log_alphas)} gates')

# Freeze everything — no training, pure inference
for p in circuit_model.parameters():
    p.requires_grad = False

# Switch to hard 0/1 binary mask mode
circuit_model.set_final_circuit_mode(True)
circuit_model.eval()
print('Circuit model ready in final (binary mask) mode.')

## Inspect the binary masks

In [ ]:
binary_masks = torch.load(os.path.join(SAVE_DIR, 'binary_masks.pt'), map_location='cpu')

total_gates = sum(m.numel() for m in binary_masks.values())
open_gates  = sum(m.sum().item() for m in binary_masks.values())
sparsity    = 1.0 - open_gates / total_gates

print(f'Total gates : {total_gates:,}')
print(f'Open  gates : {int(open_gates):,}')
print(f'Sparsity    : {sparsity:.4f} ({sparsity*100:.1f}%)')
print()

# Per-gate-type summary
from collections import defaultdict
type_stats = defaultdict(lambda: {'total': 0, 'open': 0})
for name, mask in binary_masks.items():
    for keyword in ['head_gates', 'neuron_gates', 'hidden_gates', 'output_gates',
                    'attention_block_gate', 'mlp_block_gate', 'layer_gates']:
        if keyword in name:
            type_stats[keyword]['total'] += mask.numel()
            type_stats[keyword]['open']  += int(mask.sum().item())
            break

print(f'{'Gate type':<30}  {'open':>8}  {'total':>8}  {'sparsity':>10}')
print('-' * 65)
for gtype, s in sorted(type_stats.items()):
    sp = 1 - s['open'] / s['total'] if s['total'] else 0
    print(f'{gtype:<30}  {s["open"]:>8}  {s["total"]:>8}  {sp:>10.4f}')

## Layer-wise sparsity

In [ ]:
layer_stats = defaultdict(lambda: {'total': 0, 'open': 0})
for name, mask in binary_masks.items():
    parts = name.split('.')
    layer_idx = None
    for i, p in enumerate(parts):
        if p == 'layers' and i + 1 < len(parts):
            try:
                layer_idx = int(parts[i + 1])
                break
            except ValueError:
                pass
    if layer_idx is not None:
        layer_stats[layer_idx]['total'] += mask.numel()
        layer_stats[layer_idx]['open']  += int(mask.sum().item())

layers = sorted(layer_stats.keys())
layer_sparsity = [1 - layer_stats[l]['open'] / layer_stats[l]['total'] for l in layers]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(layers, layer_sparsity, color='steelblue', edgecolor='white')
ax.axhline(sparsity, color='red', linestyle='--', label=f'Overall ({sparsity:.3f})')
ax.set_xlabel('Layer')
ax.set_ylabel('Sparsity')
ax.set_title('Layer-wise Gate Sparsity')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Load test dataset

In [ ]:
with open(os.path.join(SAVE_DIR, 'test_data.pkl'), 'rb') as f:
    test_data = pickle.load(f)

test_dataset   = IOIDatasetLlama(test_data, tokenizer)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f'Test samples: {len(test_data)}')
print('Sample keys:', list(test_data[0].keys()))

## Run evaluation — per-sample results

In [ ]:
circuit_model.eval()
full_model.eval()

records = []  # one dict per sample

with torch.no_grad():
    for batch in tqdm(test_dataloader, desc='Evaluating circuit'):
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        circuit_out = circuit_model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            corrupted_input_ids=batch.get('corrupted_input_ids'),
        )
        full_out = full_model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
        )

        B = circuit_out.logits.size(0)
        for i in range(B):
            t_start = batch['T_Start'][i].item() - 1
            t_end   = batch['T_End'][i].item() - 1
            d_start = batch['D_Start'][i].item() - 1
            d_end   = batch['D_End'][i].item() - 1

            T_len = batch['T_len'][i].item()
            D_len = batch['D_len'][i].item()
            target_tok     = batch['target_tokens'][i][:T_len]
            distractor_tok = batch['distractor_tokens'][i][:D_len]

            # Logit diff at the prediction position
            c_logit_t = circuit_out.logits[i, t_start, target_tok[0]].item()
            c_logit_d = circuit_out.logits[i, d_start, distractor_tok[0]].item()
            f_logit_t = full_out.logits[i, t_start, target_tok[0]].item()
            f_logit_d = full_out.logits[i, d_start, distractor_tok[0]].item()

            c_logit_diff = c_logit_t - c_logit_d
            f_logit_diff = f_logit_t - f_logit_d

            # KL at the IOI window
            valid_len = batch['attention_mask'][i].sum().item()
            end_pos   = min(t_end, valid_len)
            kl = 0.0
            if t_start < end_pos:
                kl = F.kl_div(
                    F.log_softmax(circuit_out.logits[i, t_start:end_pos].float(), dim=-1),
                    F.log_softmax(full_out.logits[i, t_start:end_pos].float(), dim=-1),
                    log_target=True,
                    reduction='batchmean',
                ).item()

            records.append({
                'correct_circuit': c_logit_diff > 0,
                'correct_full':    f_logit_diff > 0,
                'logit_diff_circuit': c_logit_diff,
                'logit_diff_full':    f_logit_diff,
                'kl': kl,
                'target':     tokenizer.decode(target_tok.tolist()),
                'distractor': tokenizer.decode(distractor_tok.tolist()),
                'sentence':   tokenizer.decode(batch['input_ids'][i].tolist(), skip_special_tokens=True),
            })

import pandas as pd
df = pd.DataFrame(records)

print(f"Circuit  accuracy : {df['correct_circuit'].mean():.4f}")
print(f"Full     accuracy : {df['correct_full'].mean():.4f}")
print(f"Mean KL  (circuit vs full): {df['kl'].mean():.4f}")
print(f"Mean logit-diff circuit : {df['logit_diff_circuit'].mean():.4f}")
print(f"Mean logit-diff full    : {df['logit_diff_full'].mean():.4f}")

## Distribution plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df['logit_diff_circuit'], bins=50, alpha=0.7, label='Circuit', color='steelblue')
axes[0].hist(df['logit_diff_full'],    bins=50, alpha=0.7, label='Full',    color='orange')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Logit diff (target - distractor)')
axes[0].set_title('Logit Difference Distribution')
axes[0].legend()

axes[1].hist(df['kl'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('KL divergence')
axes[1].set_title('Per-sample KL (Circuit vs Full)')

axes[2].scatter(df['logit_diff_full'], df['logit_diff_circuit'], alpha=0.3, s=10)
axes[2].axhline(0, color='red',  linestyle='--', linewidth=0.8)
axes[2].axvline(0, color='blue', linestyle='--', linewidth=0.8)
lim = max(abs(df['logit_diff_full'].max()), abs(df['logit_diff_circuit'].max())) + 1
axes[2].plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.8, label='y=x')
axes[2].set_xlabel('Full model logit-diff')
axes[2].set_ylabel('Circuit logit-diff')
axes[2].set_title('Full vs Circuit Logit Diff')
axes[2].legend()

plt.tight_layout()
plt.show()

## Failure analysis — samples the circuit gets wrong

In [ ]:
failures = df[~df['correct_circuit']].sort_values('logit_diff_circuit')
print(f'Circuit failures: {len(failures)} / {len(df)}')
failures[['sentence', 'target', 'distractor', 'logit_diff_circuit', 'logit_diff_full', 'kl']].head(10)

## Cases where circuit succeeds but full model fails

In [ ]:
circuit_only = df[df['correct_circuit'] & ~df['correct_full']]
print(f'Circuit correct, full wrong: {len(circuit_only)}')
circuit_only[['sentence', 'target', 'distractor', 'logit_diff_circuit', 'logit_diff_full']].head(10)

## High-KL samples (circuit diverges most from full model)

In [ ]:
df.sort_values('kl', ascending=False)[['sentence', 'target', 'distractor', 'kl', 'correct_circuit', 'correct_full']].head(10)

## Save results dataframe

In [ ]:
out_path = os.path.join(SAVE_DIR, 'test_results.csv')
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} rows -> {out_path}')